<div align="center">

  <h1><img src="https://raw.githubusercontent.com/auxeno/ion/main/assets/logo.png" alt="Ion" width="72"><br>Ion Tour Notebook</h1>

  [![Open in nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/auxeno/ion/blob/main/examples/ion_tour.ipynb)
  [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auxeno/ion/blob/main/examples/ion_tour.ipynb)

</div>

---

A hands-on walkthrough of [Ion](https://github.com/auxeno/ion), a simple neural network library for JAX.

In this notebook we demo the four core concepts from Ion:

1. **`Module`**: define models as immutable JAX pytrees
2. **`Param`**: mark arrays as trainable or frozen
3. **`ion.grad`**: differentiate only trainable parameters
4. **`ion.apply_updates`**: apply optimizer updates to a model

> *Click the badges above to open in **nbviewer** (fully rendered) or **Colab** (run it yourself).*

In [1]:
# !pip install git+https://github.com/auxeno/ion optax plotly

In [2]:
import jax
import jax.numpy as jnp
import optax
import plotly.graph_objects as go

import ion
from ion import nn

## The dataset

We'll learn $\sin(x)$ on $[-\pi, \pi]$, a simple regression task that runs in seconds on CPU.

In [3]:
x_train = jnp.linspace(-jnp.pi, jnp.pi, 100).reshape(-1, 1)
y_train = jnp.sin(x_train)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_train[:, 0], y=y_train[:, 0], mode="markers", name="sin(x)", marker=dict(size=6, symbol="circle", color="rgb(96,96,96)")))
fig.update_layout(title="Training data", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

## Defining a Module

Subclass `nn.Module`, declare your fields with type annotations, and write `__init__` and `__call__`. Ion registers the class as a JAX pytree and makes it immutable after `__init__`.

We'll define a custom `Linear` layer and compose an `MLP` from it. This covers the three kinds of field you'll use:

- **`nn.Param`** (`w`, `b`): wraps an array and marks it as a trainable parameter
- **Callables** (`activation`): static metadata, invisible to JAX tracing
- **Plain arrays** (`input_scale`): part of the pytree but not a `Param`, so they won't receive gradients

In [4]:
from collections.abc import Callable
from jaxtyping import Array, Float, PRNGKeyArray


class Linear(nn.Module):
    w: nn.Param
    b: nn.Param
    activation: Callable | None

    def __init__(self, in_dim: int, out_dim: int, activation=None, *, key: PRNGKeyArray):
        self.w = nn.Param(jax.nn.initializers.glorot_uniform()(key, (in_dim, out_dim)))
        self.b = nn.Param(jnp.zeros(out_dim))
        self.activation = activation

    def __call__(self, x):
        x = x @ self.w + self.b
        if self.activation is not None:
            x = self.activation(x)
        return x


class MLP(nn.Module):
    fc_1: Linear
    fc_2: Linear
    input_scale: jax.Array

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, *, key: PRNGKeyArray):
        k1, k2 = jax.random.split(key)
        self.fc_1 = Linear(in_dim, hidden_dim, activation=jax.nn.relu, key=k1)
        self.fc_2 = Linear(hidden_dim, out_dim, key=k2)
        self.input_scale = jnp.array(1.0 / jnp.pi)

    def __call__(self, x: Float[Array, "... d"]) -> Float[Array, "... d"]:
        x = x * self.input_scale
        x = self.fc_1(x)
        x = self.fc_2(x)
        return x


model = MLP(in_dim=1, hidden_dim=16, out_dim=1, key=jax.random.key(0))
model

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=True),
    b=Param(f32[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=f32[],
)

Treescope is enabled by default for interactive rendering in notebooks. Click on elements to expand or collapse them.

Ion also has a built-in text representation for terminals and logging:

In [5]:
ion.disable_treescope()
print(model)
ion.enable_treescope()

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=True),
    b=Param(f32[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=f32[],
)


## Params

Every learnable array in a model is wrapped in `nn.Param`. Params track whether they are **trainable** or **frozen** and behave like regular arrays in expressions, so no unwrapping is needed.

In [6]:
model.fc_1.w

Param(f32[1, 16], trainable=True)

In [7]:
# Params work transparently in arithmetic, no unwrapping needed
x_sample = jnp.ones((1,))
result = x_sample @ model.fc_1.w

print(f"Result shape: {result.shape}")
print(f"Result type:  {type(result).__name__}  (regular array, not Param)")

Result shape: (16,)
Result type:  ArrayImpl  (regular array, not Param)


`.params` returns the full model pytree with non-`Param` leaves set to `None`. The tree structure is preserved, including static fields like activation functions. This is what you pass to an Optax optimizer.

In [8]:
model.params

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=True),
    b=Param(f32[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=None,
)

In [9]:
print(f"Total parameters: {model.num_params:,}")

Total parameters: 49


## Modules are pytrees

Ion modules are registered as JAX pytrees, so standard `jax.tree` operations work out of the box.

In [10]:
# The leaves of the tree are Param values and plain arrays
leaves = jax.tree.leaves(model)
print(f"{len(leaves)} leaves:")
for leaf in leaves:
    print(f"  {str(leaf.dtype):8s} {str(leaf.shape)}")

5 leaves:
  float32  (1, 16)
  float32  (16,)
  float32  (16, 1)
  float32  (1,)
  float32  ()


In [11]:
# Zero all leaves
jax.tree.map(lambda p: jnp.zeros_like(p), model)

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=True),
    b=Param(f32[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=f32[],
)

## JAX transforms just work

Because modules are pytrees, `jax.jit` and `jax.vmap` work directly with no wrappers required.

In [12]:
# JIT-compile the model directly, it's just a callable pytree
jit_model = jax.jit(model)
y_hat = jit_model(x_train[:5])
print(f"JIT output shape: {y_hat.shape}")

JIT output shape: (5, 1)


In [13]:
# vmap over a batch dimension, also just works
batched_input = jax.random.normal(jax.random.key(1), (8, 1))
y_batched = jax.vmap(model)(batched_input)
print(f"vmap output shape: {y_batched.shape}")

vmap output shape: (8, 1)


## Gradients

`ion.grad` and `ion.value_and_grad` work like their JAX counterparts, but only differentiate **trainable `Param`** leaves. The returned gradient tree has the same structure as the model: trainable `Param` positions contain gradient values, while frozen `Param` positions and non-`Param` leaves become `None`.

In [14]:
def loss_fn(model, x, y):
    y_hat = model(x)
    return jnp.mean((y_hat - y) ** 2)

loss, grads = ion.value_and_grad(loss_fn)(model, x_train, y_train)
print(f"Loss: {loss:.4f}")

Loss: 0.7954


In [15]:
# Same structure as the model, with non-Param leaves set to None
grads

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=True),
    b=Param(f32[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=None,
)

## Freeze and unfreeze

You can freeze an entire model or individual layers. Frozen parameters have `trainable=False` and are skipped by `ion.grad`.

In [16]:
# Freeze the entire model
frozen_model = model.freeze()
frozen_model

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=False),
    b=Param(f32[16], trainable=False),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=False),
    b=Param(f32[1], trainable=False),
    activation=None,
  ),
  input_scale=f32[],
)

In [17]:
# Selectively freeze just the first layer
partially_frozen = model.replace(fc_1=model.fc_1.freeze())
partially_frozen

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=False),
    b=Param(f32[16], trainable=False),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=f32[],
)

In [18]:
# Frozen params become None in the gradient tree (the Param wrapper is removed entirely)
frozen_grads = ion.grad(loss_fn)(partially_frozen, x_train, y_train)
frozen_grads

MLP(
  fc_1=Linear(
    w=None,
    b=None,
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=None,
)

## Training loop

The training loop is straightforward: compute gradients with `ion.value_and_grad`, get updates from an [Optax](https://github.com/google-deepmind/optax) optimizer, and apply them with `ion.apply_updates`.

In [19]:
# Start fresh
model = MLP(in_dim=1, hidden_dim=16, out_dim=1, key=jax.random.key(0))

optimizer = optax.adam(5e-3)
opt_state = optimizer.init(model.params)


@jax.jit
def train_step(model, opt_state, x, y):
    loss, grads = ion.value_and_grad(loss_fn)(model, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, model.params)
    model = ion.apply_updates(model, updates)
    return model, opt_state, loss


losses = []
for step in range(200):
    model, opt_state, loss = train_step(model, opt_state, x_train, y_train)
    losses.append(loss.item())

print(f"Final loss: {losses[-1]:.6f}")

Final loss: 0.045831


In [20]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=losses, mode="lines", name="MSE loss", line=dict(width=5, color="#636EFA")))
fig.update_layout(title="Training loss", xaxis_title="Step", yaxis_title="Loss", height=400, template="plotly_white", yaxis_type="log")
fig.show()

In [21]:
x_eval = jnp.linspace(-jnp.pi, jnp.pi, 500).reshape(-1, 1)
x_sin = jnp.linspace(-jnp.pi, jnp.pi, 100)
y_pred = model(x_eval)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_sin, y=jnp.sin(x_sin), mode="markers", name="sin(x)", marker=dict(size=6, symbol="circle", color="rgb(96,96,96)")))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred[:, 0], mode="lines", name="Model", line=dict(width=5, color="#636EFA"), opacity=0.75))
fig.update_layout(title="Trained model vs sin(x)", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

## Model surgery

Modules are immutable after `__init__`, so you can't mutate them in place. Use `.replace()` to create a modified copy.

In [22]:
# Direct mutation is not allowed
try:
    model.fc_1 = Linear(1, 64, key=jax.random.key(99))
except AttributeError as e:
    print(f"AttributeError: {e}")

AttributeError: Cannot set attribute 'fc_1', MLP is frozen after __init__.


### Swap activation function

`activation` lives on the `Linear` layer and can be swapped with nested `.replace()` calls. Since it's static metadata, `jax.jit` handles the recompilation automatically.

In [23]:
# Swap activation on the first layer. Weights are kept, only activation changes.
tanh_model = model.replace(fc_1=model.fc_1.replace(activation=jax.nn.tanh))
opt_state_tanh = optimizer.init(tanh_model.params)

tanh_losses = []
for step in range(200):
    tanh_model, opt_state_tanh, loss = train_step(tanh_model, opt_state_tanh, x_train, y_train)
    tanh_losses.append(loss.item())

print(f"ReLU final loss: {losses[-1]:.6f}")
print(f"Tanh final loss: {tanh_losses[-1]:.6f}")

ReLU final loss: 0.045831
Tanh final loss: 0.163851


In [24]:
y_pred_tanh = tanh_model(x_eval)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_sin, y=jnp.sin(x_sin), mode="markers", name="sin(x)", marker=dict(size=6, symbol="circle", color="rgb(96,96,96)")))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred[:, 0], mode="lines", name="ReLU", line=dict(width=5, color="#636EFA"), opacity=0.75))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred_tanh[:, 0], mode="lines", name="Tanh", line=dict(width=5, color="#EF553B"), opacity=0.75))
fig.update_layout(title="Activation function comparison", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

### Resize hidden layers

We can swap in wider layers with `.replace()`. The weights are reinitialized since the dimensions change.

In [25]:
k1, k2 = jax.random.split(jax.random.key(42))
wide_model = model.replace(
    fc_1=Linear(1, 128, activation=jax.nn.relu, key=k1),
    fc_2=Linear(128, 1, key=k2),
)

print(f"Original: {model.num_params:,} params  (hidden_dim=16)")
print(f"Wide:     {wide_model.num_params:,} params  (hidden_dim=128)")

Original: 49 params  (hidden_dim=16)
Wide:     385 params  (hidden_dim=128)


In [26]:
opt_state_wide = optimizer.init(wide_model.params)

wide_losses = []
for step in range(200):
    wide_model, opt_state_wide, loss = train_step(wide_model, opt_state_wide, x_train, y_train)
    wide_losses.append(loss.item())

y_pred_wide = wide_model(x_eval)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_sin, y=jnp.sin(x_sin), mode="markers", name="sin(x)", marker=dict(size=6, symbol="circle", color="rgb(96,96,96)")))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred[:, 0], mode="lines", name="16 hidden", line=dict(width=5, color="#636EFA"), opacity=0.75))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred_wide[:, 0], mode="lines", name="128 hidden", line=dict(width=5, color="#00CC96"), opacity=0.75))
fig.update_layout(title="Effect of hidden size", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

## Checkpointing

`ion.save()` serializes array leaves and trainable flags to a `.npz` file. `ion.load()` restores them into an existing model structure, which provides the non-array fields like activation functions.

In [27]:
import tempfile, os

path = os.path.join(tempfile.gettempdir(), "model.npz")
ion.save(path, model)
print(f"Saved to {path}  ({os.path.getsize(path):,} bytes)")

Saved to /var/folders/zv/91s6yg812kz5b5j9q8ns1rb80000gn/T/model.npz  (1,864 bytes)


In [28]:
# Load into a fresh (randomly initialized) model
fresh_model = MLP(in_dim=1, hidden_dim=16, out_dim=1, key=jax.random.key(999))

print(f"Fresh model loss:  {loss_fn(fresh_model, x_train, y_train):.4f}")

loaded_model = ion.load(path, fresh_model)
print(f"Loaded model loss: {loss_fn(loaded_model, x_train, y_train):.6f}")
print(f"Weights match:     {jnp.allclose(model.fc_1.w.value, loaded_model.fc_1.w.value)}")

Fresh model loss:  0.4325
Loaded model loss: 0.045351
Weights match:     True


In [29]:
# Trainable flags are preserved through save/load
frozen_model = model.replace(fc_1=model.fc_1.freeze())
ion.save(path, frozen_model)

loaded_frozen = ion.load(path, fresh_model)
print(f"fc_1.w trainable: {loaded_frozen.fc_1.w.trainable}  (was frozen before save)")
print(f"fc_2.w trainable: {loaded_frozen.fc_2.w.trainable}  (was trainable before save)")

os.remove(path)

fc_1.w trainable: False  (was frozen before save)
fc_2.w trainable: True  (was trainable before save)


That covers Ion's core. Ion ships with standard layers (linear, convolution, attention, normalization, recurrent, pooling, and more) built on these same concepts. See the [README](https://github.com/auxeno/ion#layers) for the full list. For a complete training example, see [mnist_cnn.py](https://github.com/auxeno/ion/blob/main/examples/mnist_cnn.py).